### Explicación del Dataset

El Glass dataset es un dataset forense. Proviene de investigaciones criminológicas donde se analizaban fragmentos de vidrio encontrados en escenas del crimen.
El objetivo es clasificar el tipo de vidrio a partir de su composición química.

### Librerias necesarias para llevar acabo la actividad

In [1]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


### Imports necesarios para la actividad

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

### Cargamos del dataset

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
import numpy as np

glass = fetch_openml(name='glass', version=1, as_frame=False)

# Convertir clases de texto a números
le = LabelEncoder()
target = le.fit_transform(glass.target).reshape(-1, 1).astype(float)

dataset = np.hstack([target, glass.data])
np.savetxt('Datasets/glass.csv', dataset, delimiter=',')
print("Clases:", le.classes_)

Clases: ['build wind float' 'build wind non-float' 'containers' 'headlamps'
 'tableware' 'vehic wind float']


### Division de los datos

In [4]:
import numpy as np

dataset = np.loadtxt('Datasets/glass.csv', delimiter=',')
X = dataset[:, 1:]
y = dataset[:, 0]
print(X[0:5])
print(y[0:5])

[[ 1.51793 12.79     3.5      1.12    73.03     0.64     8.77     0.
   0.     ]
 [ 1.51643 12.16     3.52     1.35    72.89     0.57     8.53     0.
   0.     ]
 [ 1.51793 13.21     3.48     1.41    72.64     0.59     8.43     0.
   0.     ]
 [ 1.51299 14.4      1.74     1.54    74.55     0.       7.59     0.
   0.     ]
 [ 1.53393 12.3      0.       1.      70.16     0.12    16.19     0.
   0.24   ]]
[0. 5. 0. 4. 1.]


### Tranformación de los datos

In [5]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.astype(int), dtype=torch.long)
print(X[0:5])
print(y[0:5])

tensor([[ 1.5179, 12.7900,  3.5000,  1.1200, 73.0300,  0.6400,  8.7700,  0.0000,
          0.0000],
        [ 1.5164, 12.1600,  3.5200,  1.3500, 72.8900,  0.5700,  8.5300,  0.0000,
          0.0000],
        [ 1.5179, 13.2100,  3.4800,  1.4100, 72.6400,  0.5900,  8.4300,  0.0000,
          0.0000],
        [ 1.5130, 14.4000,  1.7400,  1.5400, 74.5500,  0.0000,  7.5900,  0.0000,
          0.0000],
        [ 1.5339, 12.3000,  0.0000,  1.0000, 70.1600,  0.1200, 16.1900,  0.0000,
          0.2400]])
tensor([0, 5, 0, 4, 1])


### Definimos el modelo

In [6]:
model = nn.Sequential(
    nn.Linear(9, 64),    # 9 features de entrada, capa oculta de 64
    nn.ReLU(),
    nn.Linear(64, 32),   # reducimos a 32
    nn.ReLU(),
    nn.Linear(32, 6),    # 6 clases de salida
    nn.Softmax(dim=1)    # multiclase en vez de Sigmoid
)
print(model)

Sequential(
  (0): Linear(in_features=9, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=6, bias=True)
  (5): Softmax(dim=1)
)


### Funcion de perdida

In [7]:
loss_fn = nn.CrossEntropyLoss()  # multiclase en vez de BCELoss
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Entenamos el modelo

In [8]:
n_epochs = 500
batch_size = 10
num_samples = min(len(X), len(y)) # Use the minimum length to ensure X and y are always aligned

for epoch in range(n_epochs):
  for i in range(0, num_samples, batch_size):
    # Define the end index for the current batch, ensuring it doesn't exceed num_samples
    current_batch_end = min(i + batch_size, num_samples)
    Xbatch = X[i:current_batch_end] #Partición del conjunto de datos
    ybatch = y[i:current_batch_end] #Corresponding partition for labels

    # Skip if the batch is empty (e.g., if num_samples was 0 or due to an edge case)
    if Xbatch.shape[0] == 0:
      continue

    #Forward
    y_pred = model(Xbatch)
    loss = loss_fn(y_pred, ybatch) # Calculo la pérdida
    optimizer.zero_grad()
    loss.backward()
    optimizer.step() # Actualiza pesos
  print(f'Finished epoch {epoch}, latest loss {loss}')

Finished epoch 0, latest loss 1.3051166534423828
Finished epoch 1, latest loss 1.2977491617202759
Finished epoch 2, latest loss 1.3026750087738037
Finished epoch 3, latest loss 1.7125951051712036
Finished epoch 4, latest loss 1.296994686126709
Finished epoch 5, latest loss 1.297053337097168
Finished epoch 6, latest loss 1.3000175952911377
Finished epoch 7, latest loss 1.3273805379867554
Finished epoch 8, latest loss 1.589963436126709
Finished epoch 9, latest loss 1.2982850074768066
Finished epoch 10, latest loss 1.2987916469573975
Finished epoch 11, latest loss 1.3037652969360352
Finished epoch 12, latest loss 1.3901846408843994
Finished epoch 13, latest loss 1.6195030212402344
Finished epoch 14, latest loss 1.2981557846069336
Finished epoch 15, latest loss 1.2993351221084595
Finished epoch 16, latest loss 1.3069311380386353
Finished epoch 17, latest loss 1.4230374097824097
Finished epoch 18, latest loss 1.6636453866958618
Finished epoch 19, latest loss 1.3694069385528564
Finished epoc

### Evaluamos el modelo

In [10]:
with torch.no_grad():
    y_pred = model(X)
    accuracy = (y_pred.argmax(dim=1) == y).float().mean()
    print(f"Accuracy {accuracy}")

Accuracy 0.644859790802002


In [12]:
# Sin no grad
y_pred = model(X)
accuracy = (y_pred.argmax(dim=1) == y).float().mean()
print(f"Accuracy {accuracy}")

Accuracy 0.644859790802002


### Hacer predicciones

In [13]:
prediction = model(X)
rounded = prediction.round()
print(rounded)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        ...,
        [0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]], grad_fn=<RoundBackward0>)


In [15]:
prediction = model(X).argmax(dim=1)
for i in range(10):
    print('%s --> %d (expected %d)' % (X[i].tolist(), prediction[i], y[i]))

[1.517930030822754, 12.789999961853027, 3.5, 1.1200000047683716, 73.02999877929688, 0.6399999856948853, 8.770000457763672, 0.0, 0.0] --> 0 (expected 0)
[1.5164300203323364, 12.15999984741211, 3.5199999809265137, 1.350000023841858, 72.88999938964844, 0.5699999928474426, 8.529999732971191, 0.0, 0.0] --> 0 (expected 5)
[1.517930030822754, 13.210000038146973, 3.4800000190734863, 1.409999966621399, 72.63999938964844, 0.5899999737739563, 8.430000305175781, 0.0, 0.0] --> 0 (expected 0)
[1.5129899978637695, 14.399999618530273, 1.7400000095367432, 1.5399999618530273, 74.55000305175781, 0.0, 7.590000152587891, 0.0, 0.0] --> 3 (expected 4)
[1.5339299440383911, 12.300000190734863, 0.0, 1.0, 70.16000366210938, 0.11999999731779099, 16.190000534057617, 0.0, 0.23999999463558197] --> 1 (expected 1)
[1.5165499448776245, 12.75, 2.8499999046325684, 1.440000057220459, 73.2699966430664, 0.5699999928474426, 8.789999961853027, 0.10999999940395355, 0.2199999988079071] --> 1 (expected 1)
[1.5177899599075317, 13